# Cycle Eulerien - Implementation et Etude Experimentale

Ce notebook presente l'implementation et l'etude experimentale des cycles euleriens dans les graphes non orientes.

Un **cycle eulerien** est un cycle qui passe par chaque arete du graphe exactement une fois et revient au sommet de depart.

**Theoreme d'Euler** : Un graphe connexe possede un cycle eulerien si et seulement si tous ses sommets sont de degre pair.

### Plan
1. Imports et dependances
2. Definition de la classe `Graphe` (matrice d'adjacence, degres)
3. Detection d'un cycle eulerien (connexite + parite des degres)
4. Recherche d'un cycle eulerien (algorithme de Hierholzer)
5. Generation aleatoire de graphes euleriens
6. Etude experimentale du temps de calcul

## 1. Imports et dependances

On utilise les bibliotheques suivantes :
- `random` : pour la generation aleatoire de graphes
- `time` : pour mesurer les temps de calcul
- `seaborn` et `matplotlib` : pour la visualisation des resultats

In [ ]:
import random
import time
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Classe Graphe - Initialisation et calcul des degres

La classe `Graphe` represente un graphe non oriente a l'aide d'une **matrice d'adjacence**.

Lors de l'initialisation, on calcule automatiquement :
- Le **nombre de sommets** (taille de la matrice)
- Les **degres** de chaque sommet (somme de chaque ligne)
- Le **nombre d'aretes** (somme des degres divisee par 2)

In [ ]:
class Graphe:
    def __init__(self, matrice_adjacence):
        """
        Initialise un graphe avec une matrice d'adjacence.
        """
        self.matrice_adjacence = matrice_adjacence  # Stocke la matrice d'adjacence
        self.nombre_sommets = len(matrice_adjacence)  # Calcule le nombre de sommets
        self.degres = self.calculer_degres()  # Calcule les degres des sommets
        self.nombre_aretes = sum(self.degres) // 2  # Calcule le nombre d'aretes

    def calculer_degres(self):
        """
        Calcule et retourne la liste des degres de chaque sommet.
        """
        return [sum(ligne) for ligne in self.matrice_adjacence]  # Calcule le degre de chaque sommet

## 3. Detection d'un cycle eulerien

Pour determiner si un graphe possede un cycle eulerien, on verifie deux conditions :

1. **Le graphe est connexe** : tous les sommets de degre non nul sont atteignables depuis n'importe quel autre sommet de degre non nul. On utilise un parcours en profondeur (DFS) pour verifier cela.

2. **Tous les sommets ont un degre pair** : condition necessaire et suffisante (avec la connexite) pour l'existence d'un cycle eulerien.

In [ ]:
def a_cycle_eulerien(self):
    """
    Verifie si le graphe a un cycle eulerien.
    """
    def est_connexe():
        """
        Verifie si le graphe est connexe.
        """
        def dfs(sommet, visite):
            """
            Parcourt le graphe en profondeur a partir d'un sommet donne.
            """
            visite[sommet] = True  # Marque le sommet comme visite
            for voisin in range(self.nombre_sommets):  # Pour chaque sommet voisin
                if self.matrice_adjacence[sommet][voisin] > 0 and not visite[voisin]:  # Si une arete existe et que le voisin n'est pas visite
                    dfs(voisin, visite)  # Applique DFS sur le voisin
        
        visite = [False] * self.nombre_sommets  # Initialise les sommets comme non visites
        sommet_depart = -1  # Initialisation du sommet de depart
        for i in range(self.nombre_sommets):
            if self.degres[i] > 0:  # Trouve un sommet avec un degre non nul
                sommet_depart = i
                break
        
        if sommet_depart == -1:  # Si aucun sommet avec un degre non nul n'est trouve
            return False
        
        dfs(sommet_depart, visite)  # Lance DFS a partir du sommet de depart
        
        for i in range(self.nombre_sommets):
            if self.degres[i] > 0 and not visite[i]:  # Si un sommet avec un degre non nul n'est pas visite
                return False
        return True  # Le graphe est connexe
    
    if not est_connexe():  # Verifie si le graphe est connexe
        return False
    
    for degre in self.degres:  # Verifie si chaque sommet a un degre pair
        if degre % 2 != 0:
            return False
    
    return True  # Retourne True si le graphe a un cycle eulerien

# Ajout de la methode a la classe Graphe
Graphe.a_cycle_eulerien = a_cycle_eulerien

## 4. Recherche d'un cycle eulerien - Algorithme de Hierholzer

L'**algorithme de Hierholzer** permet de trouver un cycle eulerien en temps lineaire par rapport au nombre d'aretes.

### Principe
1. On part d'un sommet de degre non nul
2. On suit les aretes en les supprimant au fur et a mesure (on travaille sur une copie de la matrice)
3. On utilise une **pile** pour gerer le parcours
4. Quand on arrive a un sommet sans arete disponible, on l'ajoute au chemin et on depile
5. Le chemin obtenu (inverse) est le cycle eulerien

In [ ]:
def trouver_cycle_eulerien(self, check=True):
    """
    Trouve et retourne un cycle eulerien dans le graphe si possible.
    """
    if check and not self.a_cycle_eulerien():  # Verifie d'abord si le graphe a un cycle eulerien
        return None
    
    g = [row[:] for row in self.matrice_adjacence]  # Cree une copie de la matrice d'adjacence
    
    def trouver_cycle(v):
        """
        Trouve un cycle a partir du sommet v en utilisant une pile pour suivre le chemin.
        """
        pile = [v]  # Initialiser la pile avec le sommet de depart
        chemin = []  # Liste pour stocker le chemin du cycle
        
        while pile:  # Tant que la pile n'est pas vide
            u = pile[-1]  # Prend le sommet au sommet de la pile
            arrete_trouvee = False  # Indicateur pour savoir si une arete a ete trouvee
            
            for i in range(len(g)):  # Cherche une arete a partir du sommet u
                if g[u][i] > 0:  # Si une arete existe
                    pile.append(i)  # Ajoute le sommet i a la pile
                    g[u][i] -= 1  # Retire l'arete du graphe
                    g[i][u] -= 1  # Retire l'arete reciproque (car le graphe est non dirige)
                    arrete_trouvee = True  # Indique qu'une arete a ete trouvee
                    break
            
            if not arrete_trouvee:  # Si aucune arete n'a ete trouvee
                chemin.append(pile.pop())  # Ajoute le sommet u au chemin
        
        return chemin  # Retourne le chemin trouve
    
    sommet_depart = 0  # Trouver un sommet de depart avec un degre non nul
    for i in range(self.nombre_sommets):
        if self.degres[i] > 0:
            sommet_depart = i
            break
    
    cycle = trouver_cycle(sommet_depart)  # Trouve le cycle eulerien a partir du sommet de depart
    
    return cycle[::-1]  # Retourner le cycle en ordre inverse pour obtenir le cycle correct

# Ajout de la methode a la classe Graphe
Graphe.trouver_cycle_eulerien = trouver_cycle_eulerien

## 5. Generation aleatoire de graphes euleriens

Pour tester notre algorithme, on genere des graphes euleriens aleatoires.

### Methode de generation
1. On cree d'abord un **cycle hamiltonien** passant par tous les sommets (garantit la connexite)
2. On ajoute ensuite des aretes aleatoires en s'assurant que chaque sommet conserve un **degre pair**

Cette approche garantit que le graphe genere est toujours connexe et eulerien.

In [ ]:
@staticmethod
def generer_graphe_eulerien(n):
    """
    Genere un graphe eulerien aleatoire avec n sommets.
    """
    if n < 3:
        raise ValueError("Le nombre de sommets doit etre au moins 3 pour generer un graphe eulerien.")
    
    matrice_adjacence = [[0] * n for _ in range(n)]  # Initialise la matrice d'adjacence

    # Creer un cycle initial
    for i in range(n):
        matrice_adjacence[i][(i + 1) % n] = 1  # Ajoute une arete vers le sommet suivant
        matrice_adjacence[(i + 1) % n][i] = 1  # Ajoute l'arete reciproque
    
    # Ajouter des aretes pour assurer que chaque sommet a un degre pair
    for i in range(n):
        voisins = [j for j in range(n) if j != i and matrice_adjacence[i][j] == 0]  # Liste des voisins possibles
        while sum(matrice_adjacence[i]) % 2 != 0:  # Tant que le degre du sommet i est impair
            j = random.choice(voisins)  # Choisit un voisin aleatoire
            matrice_adjacence[i][j] += 1  # Ajoute une arete
            matrice_adjacence[j][i] += 1  # Ajoute l'arete reciproque
            voisins.remove(j)  # Retire ce voisin de la liste
    
    return Graphe(matrice_adjacence)  # Retourne un nouveau graphe eulerien

# Ajout de la methode statique a la classe Graphe
Graphe.generer_graphe_eulerien = generer_graphe_eulerien

## Exemple d'utilisation

Generons un petit graphe eulerien avec 6 sommets et cherchons un cycle eulerien.

In [ ]:
# Exemple d'utilisation
n = 6  # Nombre de sommets
g = Graphe.generer_graphe_eulerien(n)
print("Matrice d'adjacence du graphe eulerien genere :")
for ligne in g.matrice_adjacence:
    print(ligne)

print("\nNombre de sommets :", g.nombre_sommets)
print("Nombre d'aretes :", g.nombre_aretes)
print("Degres des sommets :", g.degres)

cycle = g.trouver_cycle_eulerien()
if cycle:
    print("\nCycle eulerien trouve:", cycle)
else:
    print("\nAucun cycle eulerien n'existe dans ce graphe.")

## 6. Etude experimentale du temps de calcul

On etudie comment le temps de calcul de l'algorithme de Hierholzer evolue en fonction de la taille du graphe (nombre de sommets).

On genere des graphes euleriens de taille croissante (de 100 a 900 sommets) et on mesure le temps necessaire pour trouver un cycle eulerien dans chacun d'eux.

In [ ]:
def etude_temps_calcul():
    """
    Etudie le temps de calcul d'un cycle eulerien en fonction de la taille du graphe.
    """
    tailles = list(range(100, 1000, 100))  # Tailles des graphes a tester
    temps_calcul = []

    for taille in tailles:
        g = Graphe.generer_graphe_eulerien(taille)  # Genere un graphe eulerien aleatoire de taille specifiee
        debut = time.time()  # Enregistre le temps de debut
        g.trouver_cycle_eulerien(check=False)  # Calcule le cycle eulerien sans verification
        fin = time.time()  # Enregistre le temps de fin
        temps_calcul.append(fin - debut)  # Calcule et stocke le temps de calcul

    # Affiche les resultats sous forme de graphique
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=tailles, y=temps_calcul, marker='o')
    plt.xlabel('Nombre de sommets')
    plt.ylabel('Temps de calcul (secondes)')
    plt.title('Temps de calcul d\'un cycle eulerien en fonction de la taille du graphe')
    plt.show()

etude_temps_calcul()